# Задание 1. Закон Ципфа

В работе используются четыре частотных словаря проекта: русский и английский, каждый в двух вариантах — без лемматизации и с лемматизацией.

Для каждого словаря проверяется эмпирический закон Ципфа

$$f \cdot r = c,$$

где $f$ — абсолютная частота слова, $r$ — его ранг в частотном списке, $c$ — эмпирическая постоянная (коэффициент Ципфа).

Дополнительно в логарифмических координатах оценивается степенная зависимость

$$f(r) \approx C \cdot r^{-\alpha}.$$

Если закон Ципфа выполняется, наклон $\alpha$ близок к 1, а произведение $f \cdot r$ приблизительно постоянно в средней частотной зоне.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'freq_ru.json').exists():
    ROOT = ROOT.parent
BASE = ROOT / 'rgr2'
DATA = BASE / 'data'
FIG = BASE / 'figures'
DATA.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'legend.fontsize': 8,
    'axes.grid': True,
    'grid.alpha': 0.25,
})

DICTIONARIES = [
    ('ru_raw',   'Русский, без лемматизации',  'freq_ru.json'),
    ('ru_lemma', 'Русский, с лемматизацией',   'freq_ru_lemma.json'),
    ('en_raw',   'Английский, без лемматизации', 'freq_en.json'),
    ('en_lemma', 'Английский, с лемматизацией',  'freq_en_lemma.json'),
]

ZIPF_RANK_MIN = 100
ZIPF_RANK_MAX = 5000


def load_dictionary(filename):
    with open(ROOT / filename, encoding='utf-8') as f:
        package = json.load(f)
    items = sorted(package['dictionary'].items(), key=lambda p: (-p[1], p[0]))
    df = pd.DataFrame(items, columns=['word', 'freq'])
    df['rank'] = np.arange(1, len(df) + 1)
    df['rel_freq'] = df['freq'] / package['total_tokens']
    df['freq_x_rank'] = df['freq'] * df['rank']
    return package, df

datasets = {}
for key, label, filename in DICTIONARIES:
    package, df = load_dictionary(filename)
    datasets[key] = {'package': package, 'df': df, 'label': label, 'file': filename}
    print(f"{label}: токенов {package['total_tokens']:,}, уникальных слов {len(df):,}")


Русский, без лемматизации: токенов 4,982,552, уникальных слов 314,763


Русский, с лемматизацией: токенов 5,000,251, уникальных слов 181,819


Английский, без лемматизации: токенов 5,002,249, уникальных слов 142,303


Английский, с лемматизацией: токенов 5,002,249, уникальных слов 124,040


In [2]:
rows = []
for key, label, filename in DICTIONARIES:
    package = datasets[key]['package']
    df = datasets[key]['df']
    sub = df[(df['rank'] >= ZIPF_RANK_MIN) & (df['rank'] <= ZIPF_RANK_MAX)].copy()

    products = sub['freq_x_rank'].to_numpy(dtype=float)
    c_median = float(np.median(products))
    c_geometric = float(10 ** np.mean(np.log10(products)))
    cv = float(np.std(products, ddof=0) / np.mean(products))
    within_25 = float(np.mean(np.abs(products / c_median - 1) <= 0.25))

    x = np.log10(sub['rank'].to_numpy(dtype=float))
    y = np.log10(sub['freq'].to_numpy(dtype=float))
    alpha, intercept = np.polyfit(x, y, 1)
    y_hat = alpha * x + intercept
    r2 = 1 - np.sum((y - y_hat) ** 2) / np.sum((y - y.mean()) ** 2)

    rows.append({
        'key': key,
        'dictionary': label,
        'file': filename,
        'total_tokens': int(package['total_tokens']),
        'unique_words': int(len(df)),
        'rank_min': ZIPF_RANK_MIN,
        'rank_max': ZIPF_RANK_MAX,
        'c_median': c_median,
        'c_geometric': c_geometric,
        'cv_fr': cv,
        'within_25': within_25,
        'alpha': float(alpha),
        'intercept': float(intercept),
        'r2': float(r2),
    })

summary = pd.DataFrame(rows)
summary.to_csv(DATA / 'zipf_summary.csv', index=False, float_format='%.8f')

display(summary[['dictionary', 'total_tokens', 'unique_words', 'c_median',
                 'c_geometric', 'alpha', 'r2', 'within_25', 'cv_fr']]
        .rename(columns={
            'dictionary': 'Словарь',
            'total_tokens': 'Токенов',
            'unique_words': 'Уникальных',
            'c_median': 'c (медиана f·r)',
            'c_geometric': 'c (геом. среднее)',
            'alpha': 'Наклон α',
            'r2': 'R²',
            'within_25': 'Доля в ±25 %',
            'cv_fr': 'CV(f·r)',
        }).round(3))


,Словарь,Токенов,Уникальных,c (медиана f·r),c (геом. среднее),Наклон α,R²,Доля в ±25 %,CV(f·r)
0,"Русский, без лемматизации",4982552,314763,591965.0,560640.844,-0.844,0.999,0.915,0.117
1,"Русский, с лемматизацией",5000251,181819,656775.0,643205.677,-1.035,0.990,0.997,0.087
2,"Английский, без лемматизации",5002249,142303,581872.0,568107.344,-0.929,0.992,0.964,0.080
3,"Английский, с лемматизацией",5002249,124040,567200.0,554832.597,-1.050,0.984,0.991,0.115


## Построение графиков

На верхних панелях показаны эмпирические точки в координатах «ранг — частота» (обе оси логарифмические), теоретическая прямая $f=c/r$ и аппроксимирующая степенная зависимость. На нижних панелях — произведение $f \cdot r$ в зависимости от ранга; горизонтальная линия соответствует медианному значению $c$.

Серая полоса отмечает диапазон рангов 100–5000, в котором оценивались показатели: на краях списка закон Ципфа заведомо нарушается из-за малых частот и особенностей токенизации.


In [3]:
def make_zipf_figure(keys, title, out_path):
    fig, axes = plt.subplots(2, 2, figsize=(11.2, 8.0))
    colors = ['#1f77b4', '#d62728']

    for col, key in enumerate(keys):
        item = datasets[key]
        df = item['df']
        row = summary.set_index('key').loc[key]
        c = row['c_median']
        alpha = row['alpha']
        intercept = row['intercept']

        ax = axes[0, col]
        sub = df.iloc[::max(1, len(df) // 7000)]
        ax.scatter(sub['rank'], sub['freq'], s=4, alpha=0.30, color=colors[col],
                   linewidths=0, label='Эмпирические данные')
        r_grid = np.logspace(0, np.log10(df['rank'].max()), 400)
        ax.plot(r_grid, c / r_grid, color='black', linestyle='--', linewidth=1.4,
                label=f'$f=c/r$, c = {c:,.0f}'.replace(',', ' '))
        ax.plot(r_grid, 10 ** intercept * r_grid ** alpha, color='darkorange',
                linewidth=1.4, label=f'аппроксимация, α = {alpha:.3f}')
        ax.axvspan(ZIPF_RANK_MIN, ZIPF_RANK_MAX, color='grey', alpha=0.10)
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_title(item['label'])
        ax.set_xlabel('Ранг слова r')
        ax.set_ylabel('Частота f')
        ax.legend(loc='lower left')

        ax = axes[1, col]
        ax.scatter(sub['rank'], sub['freq_x_rank'], s=4, alpha=0.30,
                   color=colors[col], linewidths=0)
        ax.axhline(c, color='black', linestyle='--', linewidth=1.4,
                   label=f'c = {c:,.0f}'.replace(',', ' '))
        ax.axvspan(ZIPF_RANK_MIN, ZIPF_RANK_MAX, color='grey', alpha=0.10,
                   label='Диапазон оценки')
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_title(f"Произведение f·r: {item['label']}")
        ax.set_xlabel('Ранг слова r')
        ax.set_ylabel('f · r')
        ax.legend(loc='lower left')

    fig.suptitle(title, fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)
    print('Сохранено:', out_path)


make_zipf_figure(['ru_raw', 'ru_lemma'], 'Закон Ципфа для русского языка', FIG / 'fig_zipf_ru.png')
make_zipf_figure(['en_raw', 'en_lemma'], 'Закон Ципфа для английского языка', FIG / 'fig_zipf_en.png')


Сохранено: /home/dd/nlp_uni/rgr2/figures/fig_zipf_ru.png


Сохранено: /home/dd/nlp_uni/rgr2/figures/fig_zipf_en.png


In [4]:
lines = ['### Вывод по заданию 1', '']
for _, row in summary.iterrows():
    lines.append(
        f"- **{row['dictionary']}**: в диапазоне рангов {int(row['rank_min'])}–{int(row['rank_max'])} "
        f"медианное значение $c$ = {row['c_median']:.0f}, наклон $\\alpha$ = {row['alpha']:.3f}, "
        f"$R^2$ = {row['r2']:.3f}; доля слов, для которых $f\\cdot r$ отличается от $c$ не более чем на 25 %, "
        f"составляет {row['within_25']*100:.1f} %."
    )
lines += [
    '',
    'Произведение $f \\cdot r$ остаётся приблизительно постоянным в средней части частотного списка, '
    'а наклон логарифмической аппроксимации близок к −1. Это подтверждает выполнение закона Ципфа '
    'как приближённой закономерности. На краях списка наблюдаются систематические отклонения: '
    'самые частые слова дают повышенное $f \\cdot r$, а редкие слова — пониженное.',
]
display(Markdown('\n'.join(lines)))


### Вывод по заданию 1

- **Русский, без лемматизации**: в диапазоне рангов 100–5000 медианное значение $c$ = 591965, наклон $\alpha$ = -0.844, $R^2$ = 0.999; доля слов, для которых $f\cdot r$ отличается от $c$ не более чем на 25 %, составляет 91.5 %.
- **Русский, с лемматизацией**: в диапазоне рангов 100–5000 медианное значение $c$ = 656775, наклон $\alpha$ = -1.035, $R^2$ = 0.990; доля слов, для которых $f\cdot r$ отличается от $c$ не более чем на 25 %, составляет 99.7 %.
- **Английский, без лемматизации**: в диапазоне рангов 100–5000 медианное значение $c$ = 581872, наклон $\alpha$ = -0.929, $R^2$ = 0.992; доля слов, для которых $f\cdot r$ отличается от $c$ не более чем на 25 %, составляет 96.4 %.
- **Английский, с лемматизацией**: в диапазоне рангов 100–5000 медианное значение $c$ = 567200, наклон $\alpha$ = -1.050, $R^2$ = 0.984; доля слов, для которых $f\cdot r$ отличается от $c$ не более чем на 25 %, составляет 99.1 %.

Произведение $f \cdot r$ остаётся приблизительно постоянным в средней части частотного списка, а наклон логарифмической аппроксимации близок к −1. Это подтверждает выполнение закона Ципфа как приближённой закономерности. На краях списка наблюдаются систематические отклонения: самые частые слова дают повышенное $f \cdot r$, а редкие слова — пониженное.